In [8]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260615_154932"

fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))

fills

,ts,side,price,price_tick,qty,is_maker,fill_type,fill_status,inventory,mid,...,queue_ahead_at_join,snap_mid,markout_100ms,adverse_100ms,markout_500ms,adverse_500ms,markout_1000ms,adverse_1000ms,markout_5000ms,adverse_5000ms
0,1781516036443,SELL,65576.01,6557601,0.00008,True,ASK_LIFT,PARTIAL,-0.000080,65576.005,...,2.74041,65576.005,0.005,-0.005,0.005,-0.005,0.005,-0.005,-7.825,7.825
1,1781516036443,SELL,65576.01,6557601,0.08229,True,ASK_LIFT,PARTIAL,-0.082370,65576.005,...,2.74041,65576.005,0.005,-0.005,0.005,-0.005,0.005,-0.005,-7.825,7.825
2,1781516036443,SELL,65576.01,6557601,0.00008,True,ASK_LIFT,PARTIAL,-0.082450,65576.005,...,2.74041,65576.005,0.005,-0.005,0.005,-0.005,0.005,-0.005,-7.825,7.825
3,1781516036443,SELL,65576.01,6557601,0.00008,True,ASK_LIFT,PARTIAL,-0.082530,65576.005,...,2.74041,65576.005,0.005,-0.005,0.005,-0.005,0.005,-0.005,-7.825,7.825
4,1781516036443,SELL,65576.01,6557601,0.00008,True,ASK_LIFT,PARTIAL,-0.082610,65576.005,...,2.74041,65576.005,0.005,-0.005,0.005,-0.005,0.005,-0.005,-7.825,7.825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22332,1781538570508,BUY,67161.98,6716198,0.00010,True,BID_HIT,PARTIAL,-0.482154,67161.985,...,5.17096,67161.985,0.005,-0.005,0.005,-0.005,0.005,-0.005,0.005,-0.005
22333,1781538570781,BUY,67161.98,6716198,0.06018,True,BID_HIT,PARTIAL,-0.421974,67161.985,...,5.17096,67161.985,0.005,-0.005,0.005,-0.005,0.005,-0.005,0.005,-0.005
22334,1781538570860,BUY,67161.98,6716198,0.00031,True,BID_HIT,PARTIAL,-0.421664,67161.985,...,5.17096,67161.985,0.005,-0.005,0.005,-0.005,0.005,-0.005,0.005,-0.005
22335,1781538571558,BUY,67161.98,6716198,0.00010,True,BID_HIT,PARTIAL,-0.421564,67161.985,...,5.17096,67161.985,0.005,-0.005,0.005,-0.005,0.005,-0.005,0.005,-0.005


In [9]:
"""
model = predict adverse selection

                ┌──────────────┐
market ───────► │ regime model │ ───► policy params
                └──────┬───────┘
                       │
                       ▼
        ┌──────────────────────────┐
        │ alpha / fair value stack │ ───► center
        └──────────┬───────────────┘
                   │
                   ▼
        ┌──────────────────────────┐
        │ toxicity model           │ ───► risk modifiers
        └──────────┬───────────────┘
                   │
                   ▼
             execution engine

So your best toxicity model is actually:

E[future markout loss | fill now]

That is the true label your XGBoost model should learn.

Not classification. Not heuristic imbalance.
"""

df = fills

horizons = [100, 500, 1000, 5000]

feature_cols = [
    "microprice_dev",
    "order_imbalance",
    "trade_imbalance",
    "volatility",
    "spread",
    "queue_ahead_bid",
    "queue_ahead_ask",
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols].to_numpy(dtype=np.float32)
X_test = test[feature_cols].to_numpy(dtype=np.float32)

In [10]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}
models = {}

for h in horizons:

    y_train = train[f"markout_{h}ms"]
    y_test = test[f"markout_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    tox_std = np.std(train_pred)
    tox_mean_abs = np.mean(np.abs(train_pred))

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    models[h] = {
        "model": model,
        "tox_std": tox_std,
        "tox_mean_abs": tox_mean_abs
    }
    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

results_df = pd.DataFrame(results)
results_df

,100,500,1000,5000
Residual_IC,0.181789,0.202946,0.123997,-0.005726
Residual_Rank_IC,0.286049,0.181303,0.054837,0.056141
HitRate,0.576097,0.547225,0.513429,0.555506
PnLProxy,3.220883,7.417184,5.439378,0.588234
SharpeProxy,0.162796,0.103243,0.076054,0.002923


In [11]:
"""
SharpeProxy increasing with horizon

This is actually the key signal:

100ms → weak economic signal
500-1000ms → stronger signal

This tells you:

toxicity is not instantaneous micro-noise, it is short-horizon information flow

That's exactly what informed order flow looks like.

3. The important conceptual insight

You are NOT building:

a price predictor

You ARE building:

a liquidity filter

This model answers:

“Should I provide liquidity here or not?”

"""

"\nSharpeProxy increasing with horizon\n\nThis is actually the key signal:\n\n100ms → weak economic signal\n500-1000ms → stronger signal\n\nThis tells you:\n\ntoxicity is not instantaneous micro-noise, it is short-horizon information flow\n\nThat's exactly what informed order flow looks like.\n\n3. The important conceptual insight\n\nYou are NOT building:\n\na price predictor\n\nYou ARE building:\n\na liquidity filter\n\nThis model answers:\n\n“Should I provide liquidity here or not?”\n\n"

In [12]:
artifact = {
    "model": models[100]["model"],
    "feature_cols": feature_cols,
    "target": "markout_100ms",
    "horizon_ms": 100,
}

joblib.dump(artifact, "data/toxicity_model.pkl")

['data/toxicity_model.pkl']

In [13]:
artifact = {
    "name": "toxicity_model",
    "model_file": "toxicity_model_xgb.json",
    "feature_cols": feature_cols,
    "feature_dim": len(feature_cols),
    "target": "markout_100ms",
    "horizon_ms": 100
}

models[100]["model"].save_model("data/toxicity_model_xgb.json")

with open("data/toxicity_model.json", "w") as f:
    json.dump(artifact, f, indent=4)

In [14]:
def export_xgb(model_name, target, horizon_ms):
    model = models[horizon_ms]["model"]
    model.save_model(f"data/{model_name}_xgb.json")

    artifact = {
        "model_name": f"{model_name}",
        "target": target,
        "model_file": f"data/{model_name}_xgb.json",
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "feature_dim": len(feature_cols)
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f, indent=4)
    
    print(f"['data/{model_name}.json']")
    print(f"['data/{model_name}_xgb.json']")

export_xgb(model_name="toxicity_model_2", target="markout_100ms", horizon_ms=100)

['data/toxicity_model_2.json']
['data/toxicity_model_2_xgb.json']
